# ComfyUI on Google Colab

Run the cells from top to bottom. Edit the config lists to add models or custom nodes before launching.


## Optional Google Drive Mount


In [ ]:
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
# Mount or unmount Google Drive for optional persistence outside the ComfyUI install path.
from google.colab import drive

drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")


## 1. Configure Workspace, Models, and Custom Nodes


In [16]:
from pathlib import Path
import os

# Core runtime settings
WORKSPACE = "/root/comfy/ComfyUI"
COMFY_ROOT = str(Path(WORKSPACE).parent)
CUSTOM_NODES_PATH = str(Path(WORKSPACE) / "custom_nodes")
PORT = 8188
UPDATE_COMFY_UI = True
COMFYUI_LAUNCH_ARGS = "--dont-print-server --disable-auto-launch"

# Optional Hugging Face token. Leave blank to use HF_TOKEN/HUGGINGFACE_TOKEN env vars or a secure prompt.
HF_TOKEN = ""  #@param {type:"string"}

# Fill-in-the-blank model downloads.
# Use either Hugging Face fields (repo_id + filename) or a direct URL.
# dest maps under WORKSPACE/models unless it is an absolute path.
MODEL_DOWNLOADS = [
    {
        "repo_id": "black-forest-labs/FLUX.2-klein-9b-fp8",
        "filename": "flux-2-klein-9b-fp8.safetensors",
        "dest": "diffusion_models",
        "target_name": "",
    },
    {
        "repo_id": "ponpoke/flux2-klein-9b-uncensored-text-encoder",
        "filename": "flux2-klein-9b-uncensored-q8_0.gguf",
        "dest": "text_encoders",
        "target_name": "",
    },
    {
        "repo_id": "black-forest-labs/FLUX.2-klein-9B",
        "filename": "vae/diffusion_pytorch_model.safetensors",
        "dest": "vae",
        "target_name": "FLUX.2-klein-9B-vae.safetensors",
    },
    # Example direct URL entry:
    # {"url": "https://example.com/model.safetensors", "dest": "checkpoints", "target_name": "model.safetensors"},
]

# Fill-in-the-blank custom node repos. ComfyUI-Manager is installed separately and should not be added here.
CUSTOM_NODE_URLS = [
    "https://github.com/rgthree/rgthree-comfy.git",
    "https://github.com/kijai/ComfyUI-KJNodes.git",
    "https://github.com/city96/ComfyUI-GGUF.git",
    "https://github.com/yolain/ComfyUI-Easy-Use.git",
    "https://github.com/crystian/ComfyUI-Crystools.git",
    "https://github.com/jthickma/ComfyUI-Lora-Manager.git",
]

MANAGER_URL = "https://github.com/ltdrdata/ComfyUI-Manager.git"

os.environ["WORKSPACE"] = WORKSPACE
os.environ["PIP_CACHE_DIR"] = "/content/pip-cache"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:256"

Path(COMFY_ROOT).mkdir(parents=True, exist_ok=True)
Path(CUSTOM_NODES_PATH).mkdir(parents=True, exist_ok=True)

print(f"WORKSPACE: {WORKSPACE}")
print(f"Custom nodes: {CUSTOM_NODES_PATH}")
print(f"Model entries: {len(MODEL_DOWNLOADS)}")
print(f"Custom node entries: {len(CUSTOM_NODE_URLS)}")


WORKSPACE: /root/comfy/ComfyUI
Custom nodes: /root/comfy/ComfyUI/custom_nodes
Model entries: 3
Custom node entries: 6


## 2. Install or Update ComfyUI


In [17]:
from pathlib import Path
import subprocess
import sys

WORKSPACE_PATH = Path(WORKSPACE)
COMFY_ROOT_PATH = Path(COMFY_ROOT)
COMFY_ROOT_PATH.mkdir(parents=True, exist_ok=True)


def run(cmd, *, cwd=None, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)


main_py = WORKSPACE_PATH / "main.py"
requirements = WORKSPACE_PATH / "requirements.txt"

if not WORKSPACE_PATH.exists() or not any(WORKSPACE_PATH.iterdir()):
    print("No ComfyUI install found. Cloning ComfyUI into WORKSPACE...")
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI.git", str(WORKSPACE_PATH)])
elif (WORKSPACE_PATH / ".git").exists():
    print("ComfyUI git checkout exists at WORKSPACE.")
else:
    print("Existing non-git ComfyUI directory found at WORKSPACE. Validating it instead of recloning.")

if UPDATE_COMFY_UI and (WORKSPACE_PATH / ".git").exists():
    print("Updating ComfyUI with --ff-only...")
    run(["git", "pull", "--ff-only"], cwd=WORKSPACE_PATH)
elif UPDATE_COMFY_UI:
    print("Skipping git update because WORKSPACE is not a git checkout.")

if not main_py.exists():
    raise FileNotFoundError(f"ComfyUI install is incomplete: {main_py} was not found")
if not requirements.exists():
    raise FileNotFoundError(f"ComfyUI requirements file was not found: {requirements}")

print("Installing ComfyUI Python requirements...")
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])

print("ComfyUI install verified:", main_py)
%cd {WORKSPACE}


ComfyUI git checkout exists at WORKSPACE.
Updating ComfyUI with --ff-only...
$ git pull --ff-only
Installing ComfyUI Python requirements...
$ /usr/bin/python3 -m pip install -q --upgrade pip
$ /usr/bin/python3 -m pip install -q -r /root/comfy/ComfyUI/requirements.txt
ComfyUI install verified: /root/comfy/ComfyUI/main.py
/root/comfy/ComfyUI


## 3. Install ComfyUI-Manager


In [18]:
from pathlib import Path
import subprocess
import sys

CUSTOM_NODES = Path(CUSTOM_NODES_PATH)
MANAGER_PATH = CUSTOM_NODES / "ComfyUI-Manager"
CUSTOM_NODES.mkdir(parents=True, exist_ok=True)


def run(cmd, *, cwd=None, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)


if not (Path(WORKSPACE) / "main.py").exists():
    raise FileNotFoundError(f"Install ComfyUI first; missing {Path(WORKSPACE) / 'main.py'}")

if not (MANAGER_PATH / ".git").exists():
    if MANAGER_PATH.exists() and any(MANAGER_PATH.iterdir()):
        raise RuntimeError(f"{MANAGER_PATH} exists but is not a git checkout. Move it aside or delete it before repair.")
    print("Cloning ComfyUI-Manager...")
    run(["git", "clone", MANAGER_URL, str(MANAGER_PATH)])
else:
    print("Updating ComfyUI-Manager...")
    run(["git", "pull", "--ff-only"], cwd=MANAGER_PATH)

manager_requirements = MANAGER_PATH / "requirements.txt"
if manager_requirements.exists():
    print("Installing ComfyUI-Manager requirements...")
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(manager_requirements)])
else:
    print("ComfyUI-Manager has no requirements.txt; skipping requirement install.")

if not MANAGER_PATH.exists():
    raise FileNotFoundError(f"ComfyUI-Manager was not installed at {MANAGER_PATH}")

print("ComfyUI-Manager installed:", MANAGER_PATH)
print("If ComfyUI is already running, stop that launch cell and restart ComfyUI so Manager loads.")


Cloning ComfyUI-Manager...
$ git clone https://github.com/ltdrdata/ComfyUI-Manager.git /root/comfy/ComfyUI/custom_nodes/ComfyUI-Manager
Installing ComfyUI-Manager requirements...
$ /usr/bin/python3 -m pip install -q -r /root/comfy/ComfyUI/custom_nodes/ComfyUI-Manager/requirements.txt
ComfyUI-Manager installed: /root/comfy/ComfyUI/custom_nodes/ComfyUI-Manager
If ComfyUI is already running, stop that launch cell and restart ComfyUI so Manager loads.


## Repair Manager In Current Runtime


In [ ]:
# Run this cell if the ComfyUI web UI is already open but Manager is missing.
from pathlib import Path
import subprocess
import sys

WORKSPACE = globals().get("WORKSPACE", "/root/comfy/ComfyUI")
CUSTOM_NODES_PATH = globals().get("CUSTOM_NODES_PATH", str(Path(WORKSPACE) / "custom_nodes"))
MANAGER_URL = globals().get("MANAGER_URL", "https://github.com/ltdrdata/ComfyUI-Manager.git")
MANAGER_PATH = Path(CUSTOM_NODES_PATH) / "ComfyUI-Manager"
Path(CUSTOM_NODES_PATH).mkdir(parents=True, exist_ok=True)


def run(cmd, *, cwd=None, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)


if not (MANAGER_PATH / ".git").exists():
    if MANAGER_PATH.exists() and any(MANAGER_PATH.iterdir()):
        raise RuntimeError(f"{MANAGER_PATH} exists but is not a git checkout. Move it aside or delete it before repair.")
    run(["git", "clone", MANAGER_URL, str(MANAGER_PATH)])
else:
    run(["git", "pull", "--ff-only"], cwd=MANAGER_PATH)

requirements = MANAGER_PATH / "requirements.txt"
if requirements.exists():
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)])

print("ComfyUI-Manager repair complete:", MANAGER_PATH)
print("Stop the running ComfyUI launch cell, then run the Cloudflare launch cell again. Manager loads only on ComfyUI startup.")


## 4. Download Models


In [19]:
from pathlib import Path
import getpass
import os
import shutil
import subprocess
import sys
from urllib.parse import urlparse

run_pip = [sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub[cli]", "hf_transfer"]
print("Installing Hugging Face download helpers...")
subprocess.run(run_pip, check=True)

MODEL_BASE = Path(WORKSPACE) / "models"
MODEL_BASE.mkdir(parents=True, exist_ok=True)


def resolve_model_dir(dest):
    if not dest:
        raise ValueError("Each model entry needs a dest value")
    dest_path = Path(dest)
    if dest_path.is_absolute():
        return dest_path
    return MODEL_BASE / dest


def filename_from_url(url):
    path = urlparse(url).path
    name = Path(path).name
    if not name:
        raise ValueError(f"Could not infer filename from URL: {url}")
    return name


def run(cmd, *, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(cmd, check=check, text=True)


def get_hf_token():
    token = (HF_TOKEN or os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN") or "").strip()
    if token:
        return token
    try:
        return getpass.getpass("Enter your Hugging Face token, or press Enter if none is needed: ").strip()
    except Exception:
        return input("Enter your Hugging Face token, or press Enter if none is needed: ").strip()


def download_direct_url(entry, target):
    url = entry.get("url", "").strip()
    if not url:
        raise ValueError("Direct download entry is missing url")
    if shutil.which("wget"):
        run(["wget", "-c", url, "-O", str(target)])
    else:
        import urllib.request
        print(f"Downloading {url} -> {target}")
        urllib.request.urlretrieve(url, target)


def download_hf(entry, target_dir, target):
    repo_id = entry.get("repo_id", "").strip()
    filename = entry.get("filename", "").strip()
    if not repo_id or not filename:
        raise ValueError("Hugging Face entries need repo_id and filename")
    token = get_hf_token()
    cmd = ["hf", "download", repo_id, filename, "--local-dir", str(target_dir)]
    if token:
        cmd.extend(["--token", token])
    run(cmd)

    downloaded = target_dir / filename
    if downloaded.exists() and downloaded != target:
        target.parent.mkdir(parents=True, exist_ok=True)
        downloaded.replace(target)
        parent = downloaded.parent
        while parent != target_dir:
            try:
                parent.rmdir()
            except OSError:
                break
            parent = parent.parent


successes = []
failures = []
for entry in MODEL_DOWNLOADS:
    try:
        target_dir = resolve_model_dir(entry.get("dest", ""))
        target_dir.mkdir(parents=True, exist_ok=True)
        target_name = entry.get("target_name") or entry.get("filename") or filename_from_url(entry.get("url", ""))
        target = target_dir / Path(target_name).name

        if target.exists() and target.stat().st_size > 0:
            print(f"Already present: {target}")
            successes.append(str(target))
            continue

        if entry.get("url"):
            download_direct_url(entry, target)
        else:
            download_hf(entry, target_dir, target)

        if not target.exists() or target.stat().st_size == 0:
            raise RuntimeError(f"Download did not create a non-empty file: {target}")
        print(f"Ready: {target}")
        successes.append(str(target))
    except Exception as exc:
        failures.append((entry, exc))
        print(f"FAILED model entry {entry}: {exc}")

print(f"Model downloads complete: {len(successes)} succeeded, {len(failures)} failed")
if failures:
    raise RuntimeError("One or more model downloads failed; see messages above")


Installing Hugging Face download helpers...
Already present: /root/comfy/ComfyUI/models/diffusion_models/flux-2-klein-9b-fp8.safetensors
Already present: /root/comfy/ComfyUI/models/text_encoders/flux2-klein-9b-uncensored-q8_0.gguf
Already present: /root/comfy/ComfyUI/models/vae/FLUX.2-klein-9B-vae.safetensors
Model downloads complete: 3 succeeded, 0 failed


## 5. Install Custom Nodes


In [20]:
from pathlib import Path
from urllib.parse import urlparse
import subprocess
import sys

CUSTOM_NODES = Path(CUSTOM_NODES_PATH)
CUSTOM_NODES.mkdir(parents=True, exist_ok=True)
MANAGER_CANONICAL = "https://github.com/ltdrdata/ComfyUI-Manager.git"


def run(cmd, *, cwd=None, check=True):
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)


def repo_folder_name(url):
    path = urlparse(url).path.rstrip("/")
    name = Path(path).name
    if name.endswith(".git"):
        name = name[:-4]
    if not name:
        raise ValueError(f"Could not derive folder name from repo URL: {url}")
    return name


successes = []
failures = []
for url in CUSTOM_NODE_URLS:
    url = url.strip()
    if not url:
        continue
    if url.rstrip("/").lower() == MANAGER_CANONICAL[:-4].lower() or url.rstrip("/").lower() == MANAGER_CANONICAL.lower():
        print("Skipping ComfyUI-Manager in CUSTOM_NODE_URLS; it is installed by the Manager cell.")
        continue

    try:
        node_path = CUSTOM_NODES / repo_folder_name(url)
        if (node_path / ".git").exists():
            print(f"Updating {node_path.name}")
            result = run(["git", "pull", "--ff-only"], cwd=node_path)
            if result.stdout:
                print(result.stdout.strip())
            if result.stderr:
                print(result.stderr.strip())
        elif node_path.exists() and any(node_path.iterdir()):
            raise RuntimeError(f"{node_path} exists but is not a git checkout")
        else:
            print(f"Cloning {node_path.name}")
            try:
                run(["git", "clone", "--depth", "1", url, str(node_path)])
            except subprocess.CalledProcessError:
                print("Shallow clone failed; retrying full clone...")
                run(["git", "clone", url, str(node_path)])

        requirements = node_path / "requirements.txt"
        if requirements.exists():
            print(f"Installing requirements for {node_path.name}")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], check=True)
        else:
            print(f"No requirements.txt for {node_path.name}")
        successes.append(node_path.name)
    except Exception as exc:
        failures.append((url, exc))
        print(f"FAILED custom node {url}: {exc}")

print(f"Custom node install complete: {len(successes)} succeeded, {len(failures)} failed")
if failures:
    raise RuntimeError("One or more custom node installs failed; see messages above")

print("Restart ComfyUI after installing or updating custom nodes.")


Cloning rgthree-comfy
$ git clone --depth 1 https://github.com/rgthree/rgthree-comfy.git /root/comfy/ComfyUI/custom_nodes/rgthree-comfy
Installing requirements for rgthree-comfy
Cloning ComfyUI-KJNodes
$ git clone --depth 1 https://github.com/kijai/ComfyUI-KJNodes.git /root/comfy/ComfyUI/custom_nodes/ComfyUI-KJNodes
Installing requirements for ComfyUI-KJNodes
Cloning ComfyUI-GGUF
$ git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git /root/comfy/ComfyUI/custom_nodes/ComfyUI-GGUF
Installing requirements for ComfyUI-GGUF
Cloning ComfyUI-Easy-Use
$ git clone --depth 1 https://github.com/yolain/ComfyUI-Easy-Use.git /root/comfy/ComfyUI/custom_nodes/ComfyUI-Easy-Use
Installing requirements for ComfyUI-Easy-Use
Cloning ComfyUI-Crystools
$ git clone --depth 1 https://github.com/crystian/ComfyUI-Crystools.git /root/comfy/ComfyUI/custom_nodes/ComfyUI-Crystools
Installing requirements for ComfyUI-Crystools
Updating ComfyUI-Lora-Manager
$ git pull --ff-only
Already up to date.
Installing

## 6. Verify Install


In [ ]:
from pathlib import Path

workspace = Path(WORKSPACE)
manager = Path(CUSTOM_NODES_PATH) / "ComfyUI-Manager"
checks = {
    "ComfyUI main.py": workspace / "main.py",
    "ComfyUI custom_nodes": Path(CUSTOM_NODES_PATH),
    "ComfyUI-Manager": manager,
    "models": workspace / "models",
}

for label, path in checks.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:7} {label}: {path}")

if not (workspace / "main.py").exists():
    raise FileNotFoundError("ComfyUI main.py is missing; rerun the ComfyUI install cell")
if not manager.exists():
    raise FileNotFoundError("ComfyUI-Manager is missing; run the Manager install or repair cell")

print("\nInstalled custom nodes:")
for path in sorted(Path(CUSTOM_NODES_PATH).iterdir()):
    if path.is_dir():
        print("-", path.name)

print("\nModel files:")
for folder in ["checkpoints", "diffusion_models", "text_encoders", "vae", "loras"]:
    model_dir = workspace / "models" / folder
    if model_dir.exists():
        files = [p for p in model_dir.rglob("*") if p.is_file()]
        print(f"- {folder}: {len(files)} files")


## 7. Launch ComfyUI with Cloudflare


In [ ]:
from pathlib import Path
import os
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.request

WORKSPACE_PATH = Path(WORKSPACE)
PORT = int(PORT)
LOCAL_URL = f"http://127.0.0.1:{PORT}"
COMFY_LOG = Path("/tmp/comfyui.log")
CLOUDFLARED_LOG = Path("/tmp/cloudflared.log")
PORT_TIMEOUT_SECONDS = 180
HTTP_TIMEOUT_SECONDS = 60
TUNNEL_TIMEOUT_SECONDS = 90


def tail(path, lines=80):
    if not path.exists():
        return ""
    data = path.read_text(errors="replace").splitlines()
    return "\n".join(data[-lines:])


def ensure_cloudflared():
    if shutil.which("cloudflared"):
        print("cloudflared is already installed")
        return
    deb_path = "/tmp/cloudflared-linux-amd64.deb"
    print("Installing cloudflared...")
    subprocess.run([
        "wget",
        "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb",
        "-O",
        deb_path,
    ], check=True)
    subprocess.run(["dpkg", "-i", deb_path], check=True)


def wait_for_port(host, port, timeout):
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(2)
            if sock.connect_ex((host, port)) == 0:
                return True
        time.sleep(1)
    return False


def wait_for_http(url, timeout):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if 200 <= response.status < 500:
                    return True
        except Exception as exc:
            last_error = exc
        time.sleep(2)
    print(f"Last local HTTP check error: {last_error}")
    return False


def stream_process(process, log_path, prefix, url_holder=None):
    tunnel_pattern = re.compile(r"https://[-a-zA-Z0-9.]+trycloudflare.com")
    with log_path.open("w", encoding="utf-8", errors="replace") as log:
        for line in process.stdout:
            log.write(line)
            log.flush()
            if url_holder is not None:
                match = tunnel_pattern.search(line)
                if match:
                    url_holder["url"] = match.group(0)
            if prefix == "cloudflared" or "error" in line.lower() or "traceback" in line.lower():
                print(f"[{prefix}] {line.rstrip()}")


if not (WORKSPACE_PATH / "main.py").exists():
    raise FileNotFoundError(f"Missing {WORKSPACE_PATH / 'main.py'}; run the ComfyUI install cell first")
if not (Path(CUSTOM_NODES_PATH) / "ComfyUI-Manager").exists():
    raise FileNotFoundError("ComfyUI-Manager is missing; run the Manager install or repair cell before launch")

ensure_cloudflared()

for log_path in [COMFY_LOG, CLOUDFLARED_LOG]:
    try:
        log_path.unlink()
    except FileNotFoundError:
        pass

launch_args = COMFYUI_LAUNCH_ARGS.split()
comfy_cmd = [sys.executable, "main.py", *launch_args, "--listen", "127.0.0.1", "--port", str(PORT)]
print("Starting ComfyUI:", " ".join(comfy_cmd))
comfy_proc = subprocess.Popen(
    comfy_cmd,
    cwd=str(WORKSPACE_PATH),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
threading.Thread(target=stream_process, args=(comfy_proc, COMFY_LOG, "comfyui"), daemon=True).start()

try:
    print(f"Waiting up to {PORT_TIMEOUT_SECONDS}s for ComfyUI port {PORT}...")
    if not wait_for_port("127.0.0.1", PORT, PORT_TIMEOUT_SECONDS):
        print(tail(COMFY_LOG))
        raise TimeoutError(f"ComfyUI did not open port {PORT}")

    print(f"Waiting up to {HTTP_TIMEOUT_SECONDS}s for local HTTP health...")
    if not wait_for_http(LOCAL_URL, HTTP_TIMEOUT_SECONDS):
        print(tail(COMFY_LOG))
        raise TimeoutError(f"ComfyUI port opened but {LOCAL_URL} did not answer HTTP checks")

    print("ComfyUI is healthy locally. Starting Cloudflare tunnel...")
    tunnel_holder = {}
    cloudflared_proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", LOCAL_URL, "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    threading.Thread(target=stream_process, args=(cloudflared_proc, CLOUDFLARED_LOG, "cloudflared", tunnel_holder), daemon=True).start()

    deadline = time.time() + TUNNEL_TIMEOUT_SECONDS
    while time.time() < deadline and not tunnel_holder.get("url"):
        if cloudflared_proc.poll() is not None:
            break
        time.sleep(1)

    public_url = tunnel_holder.get("url")
    if not public_url:
        print(tail(CLOUDFLARED_LOG))
        raise TimeoutError("cloudflared did not print a trycloudflare.com URL before timeout")

    print("\nThis is the URL to access ComfyUI:", public_url)
    print("Leave this cell running. That is normal: it owns the ComfyUI server and Cloudflare tunnel.")
    print("Use Runtime > Interrupt execution when you want to stop ComfyUI.\n")

    while True:
        if comfy_proc.poll() is not None:
            print(tail(COMFY_LOG))
            raise RuntimeError("ComfyUI exited unexpectedly")
        if cloudflared_proc.poll() is not None:
            print(tail(CLOUDFLARED_LOG))
            raise RuntimeError("cloudflared exited unexpectedly")
        time.sleep(2)
except KeyboardInterrupt:
    print("Stopping ComfyUI and cloudflared...")
finally:
    for proc_name in ["cloudflared_proc", "comfy_proc"]:
        proc = locals().get(proc_name)
        if proc and proc.poll() is None:
            proc.terminate()


cloudflared is already installed
Starting ComfyUI: /usr/bin/python3 main.py --dont-print-server --disable-auto-launch --listen 127.0.0.1 --port 8188
Waiting up to 180s for ComfyUI port 8188...
